<a href="https://colab.research.google.com/github/saskiaalifah/SaskiaAlifah_2411531002_ML2526/blob/main/Praktikum5/ValidationTechniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import library

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

load dataset

In [14]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/ML/Dataset/advertising.csv"
df = pd.read_csv(DATA_PATH)

X = df.drop("Sales", axis=1)
y = df["Sales"]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Implementasikan Holdout Validation menggunakan Train-Test Split

In [15]:
# === Split Data (Holdout Method) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Inisialisasi & Training Model ===
model = LinearRegression()
model.fit(X_train, y_train)

# === Prediksi Data Train & Test ===
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# === Evaluasi Model (RMSE) ===
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

# === Output Hasil Evaluasi ===
print("=== Holdout Method ===")
print("RMSE Train:", rmse_train)
print("RMSE Test :", rmse_test)


=== Holdout Method ===
RMSE Train: 1.635892005537856
RMSE Test : 1.7052146229349223


K-Fold Manual

In [16]:
# === Inisialisasi K-Fold (5 Fold) ===
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# === List untuk Simpan RMSE ===
rmse_list = []

# === Looping Tiap Fold ===
for train_index, test_index in kf.split(X):
    # === Split Data Fold ===
    X_train_k, X_test_k = X.iloc[train_index], X.iloc[test_index]
    y_train_k, y_test_k = y.iloc[train_index], y.iloc[test_index]

    # === Inisialisasi & Training Model ===
    model_k = LinearRegression()
    model_k.fit(X_train_k, y_train_k)

    # === Prediksi Data Test Fold ===
    y_pred_k = model_k.predict(X_test_k)

    # === Evaluasi RMSE Fold ===
    rmse_k = np.sqrt(mean_squared_error(y_test_k, y_pred_k))
    rmse_list.append(rmse_k)

# === Output Hasil Evaluasi ===
print("\n=== K-Fold Manual (5 Fold) ===")
print("RMSE tiap fold:", rmse_list)
print("RMSE rata-rata:", np.mean(rmse_list))



=== K-Fold Manual (5 Fold) ===
RMSE tiap fold: [np.float64(1.705214622934923), np.float64(1.5406931704228382), np.float64(1.6280612215351888), np.float64(1.629171920588974), np.float64(1.8759540208373862)]
RMSE rata-rata: 1.6758189912638621


K-Fold Menggunakan cross_val_score

In [17]:
# === Evaluasi Model dengan cross_val_score ===
cv_scores = np.sqrt(
    -cross_val_score(
        LinearRegression(),
        X,
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
)

# === Output Hasil Evaluasi ===
print("\n=== K-Fold dengan cross_val_score ===")
print("RMSE tiap fold:", cv_scores)
print("RMSE rata-rata:", cv_scores.mean())



=== K-Fold dengan cross_val_score ===
RMSE tiap fold: [1.7481616  1.42364344 1.36053376 2.17264645 1.62386598]
RMSE rata-rata: 1.6657702460059216


*TUGAS PRAKTIKUM 4*

### 2. Analisis Hasil Holdout Validation

Berdasarkan hasil Holdout Validation yang telah dilakukan:

**a. Perbandingan RMSE Train dan RMSE Test**

*   **RMSE Train:** `1.6359`
*   **RMSE Test:** `1.7052`

Terlihat bahwa nilai RMSE Train (**`1.6359`**) sedikit lebih kecil dibandingkan RMSE Test (**`1.7052`**). Perbedaan ini tidak terlalu signifikan, yang mengindikasikan bahwa model tidak terlalu "bias" terhadap data pelatihan.

**b. Indikasi Overfitting atau Underfitting**

Dalam kasus ini, model tidak menunjukkan tanda-tanda **overfitting** yang parah karena perbedaan antara RMSE Train dan RMSE Test tidak terlalu besar. Overfitting akan terjadi jika RMSE Train jauh lebih kecil daripada RMSE Test, menandakan model terlalu "menghafal" data pelatihan dan buruk dalam memprediksi data baru.

Demikian pula, model juga tidak menunjukkan tanda-tanda **underfitting**. Underfitting akan terjadi jika kedua nilai RMSE (Train dan Test) sangat tinggi, yang berarti model tidak cukup kompleks untuk menangkap pola dalam data, baik pada data pelatihan maupun data pengujian.

Secara keseluruhan, model ini tampak **cukup seimbang** dengan kinerja yang konsisten pada data pelatihan dan pengujian.

### Analisis Pengaruh K-Fold Cross Validation

Berdasarkan hasil K-Fold Cross Validation yang telah dilakukan, kita dapat menganalisis beberapa aspek:

**a. Perbandingan RMSE Holdout vs RMSE K-Fold**

*   **RMSE Holdout (Test):** `1.7052` (dari metode `train_test_split`)
*   **RMSE K-Fold (Rata-rata):** `1.6658` (dari `cross_val_score`)

Terlihat bahwa nilai RMSE rata-rata dari K-Fold Cross Validation (**`1.6658`**) sedikit lebih rendah dibandingkan dengan RMSE dari Holdout Validation (**`1.7052`**). Ini mengindikasikan bahwa model, ketika dievaluasi secara lebih menyeluruh melalui berbagai subset data, memberikan kinerja yang sedikit lebih baik atau setidaknya tidak lebih buruk dibandingkan dengan evaluasi tunggal Holdout.

**b. Apakah K-Fold memberikan hasil yang lebih stabil?**

Untuk menilai stabilitas, kita melihat pada variasi hasil di setiap fold. Dengan RMSE rata-rata K-Fold sebesar `1.6658` dan nilai RMSE per fold yang bervariasi (`[1.7482, 1.4236, 1.3605, 2.1726, 1.6239]`), K-Fold memberikan gambaran yang lebih komprehensif tentang stabilitas model.

Dibandingkan dengan Holdout yang hanya memberikan satu nilai RMSE test, K-Fold memberikan rentang kinerja yang menunjukkan bagaimana model bereaksi terhadap perbedaan data pelatihan dan pengujian. Rentang nilai RMSE (dari sekitar 1.36 hingga 2.17) menunjukkan adanya variasi, namun nilai rata-ratanya cukup konsisten. Ini berarti K-Fold memberikan estimasi kinerja model yang lebih *robust* dan stabil karena mengurangi ketergantungan pada satu pembagian data acak.

**c. Bagaimana variasi kinerja model pada masing-masing fold?**

Nilai RMSE pada masing-masing fold adalah sebagai berikut:

*   **Fold 1:** `1.7482`
*   **Fold 2:** `1.4236`
*   **Fold 3:** `1.3605`
*   **Fold 4:** `2.1726`
*   **Fold 5:** `1.6239`

Ada variasi yang cukup signifikan antar fold. Fold 3 (`1.3605`) menunjukkan kinerja terbaik (RMSE terendah), sedangkan Fold 4 (`2.1726`) menunjukkan kinerja terburuk (RMSE tertinggi). Variasi ini normal dalam K-Fold Cross Validation dan menunjukkan bahwa model mungkin lebih baik dalam memprediksi pada subset data tertentu dibandingkan yang lain.

Variasi ini juga menyoroti pentingnya K-Fold, karena jika hanya menggunakan satu pembagian (seperti Holdout), kita mungkin mendapatkan hasil yang sangat baik (jika kebetulan pembagiannya mirip dengan Fold 3) atau sangat buruk (jika mirip dengan Fold 4), tanpa mengetahui rentang kinerja sebenarnya.